<a href="https://colab.research.google.com/github/acapodanno/assistant-ai/blob/main/GreenThumb_Assistant_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GreenThumb Marketplace - AI Customer Support Assistant


## 1. Installazione Dipendenze

In [3]:
!pip install -q langchain langchain-openai langchain-chroma chromadb deepeval ragas pandas loguru pydantic litellm datasets python-dotenv rank-bm25 fastapi uvicorn chainlit

## 2. Configurazione Chiave API OpenAI

In [4]:
import os
from google.colab import userdata


os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## 3. Core Engine (`main.py`)

Scrive il modulo centrale contenente i Modelli, la pipeline RAG, i Tool e il `GreenThumbAgent` basato su ReAct loop.

In [ ]:
#!/usr/bin/env python3
import os
import json
import pickle
from enum import Enum
from datetime import datetime
from typing import Optional, List
from dotenv import load_dotenv
from loguru import logger

from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from litellm import completion

load_dotenv()

class IssueType(str, Enum):
    MANCATA_CONSEGNA = "MANCATA_CONSEGNA"
    PRODOTTO_DANNEGGIATO = "PRODOTTO_DANNEGGIATO"
    RESO = "RESO"
    RIMBORSO = "RIMBORSO"
    ALTRO = "ALTRO"

class TicketStatus(str, Enum):
    OPEN = "OPEN"
    IN_PROGRESS = "IN_PROGRESS"
    CLOSED = "CLOSED"

class CustomerInfo(BaseModel):
    name: str
    email: str
    phone: str

class OrderDetails(BaseModel):
    products: list[dict] = Field(default_factory=list)
    total: Optional[float] = None
    shipping_address: Optional[str] = None
    carrier: Optional[str] = None
    tracking_number: Optional[str] = None

class Ticket(BaseModel):
    ticket_id: int
    customer: CustomerInfo
    order_id: str
    issue_type: IssueType
    issue_description: str
    order_details: OrderDetails = Field(default_factory=OrderDetails)
    status: TicketStatus = TicketStatus.OPEN
    created_at: datetime = Field(default_factory=datetime.now)

class AgentResponse(BaseModel):
    answer: str = Field(description="La risposta finale per il cliente.")
    confidence: str = Field(default="high", description="Il livello di confidenza.")
    sources: list[str] = Field(default_factory=list, description="Lista delle fonti esatte")
    needs_human: bool = Field(default=False, description="True se serve supporto umano.")

class ChatRequest(BaseModel):
    message: str

BM25_PATH = "./chroma_db/bm25_index.pkl"
LOADER_MAP = {
    ".md":   lambda path: TextLoader(file_path=path, encoding="utf-8"),
    ".txt":  lambda path: TextLoader(file_path=path, encoding="utf-8"),
    ".pdf":  lambda path: PyPDFLoader(file_path=path),
}

def retrieve_documents() -> list[Document]:
    kb_root = "./data/knowledge_base"
    documents = []
    for root, _dirs, files in os.walk(kb_root):
        for filename in files:
            filepath = os.path.join(root, filename)
            ext = os.path.splitext(filename)[1].lower()
            loader_factory = LOADER_MAP.get(ext)
            if loader_factory is None: continue
            loader = loader_factory(filepath)
            docs = loader.load()
            category = os.path.basename(root)
            for doc in docs:
                doc.metadata["file_type"] = ext.lstrip(".")
                doc.metadata["source"] = filepath
                doc.metadata["category"] = category
            documents.extend(docs)
    return documents

def chunk_documents(documents: list[Document]) -> list[Document]:
    chunked = []
    for doc in documents:
        file_type = doc.metadata.get("file_type", "txt")
        separators = ["\n# ", "\n## ", "\n### ", "\n\n", "\n", " ", ""] if file_type == "md" else ["\n\n", "\n", " ", ""]
        splitter = RecursiveCharacterTextSplitter(separators=separators, chunk_size=800, chunk_overlap=120, length_function=len)
        chunked.extend(splitter.split_documents([doc]))
    return chunked

def build_vector_store(chunked_documents: list[Document]):
    import chromadb
    client = chromadb.PersistentClient(path="./chroma_db")
    try: client.delete_collection("greenthumb_kb")
    except: pass
    return Chroma.from_documents(documents=chunked_documents, embedding=OpenAIEmbeddings(model="text-embedding-3-large"), persist_directory="./chroma_db", collection_name="greenthumb_kb")

def build_sparse_retriever(chunked_documents: list[Document]) -> BM25Retriever:
    retriever = BM25Retriever.from_documents(chunked_documents)
    retriever.k = 3
    os.makedirs("./chroma_db", exist_ok=True)
    with open(BM25_PATH, "wb") as f: pickle.dump(retriever, f)
    return retriever

def run_ingestion():
    docs = retrieve_documents()
    chunks = chunk_documents(docs)
    build_vector_store(chunks)
    build_sparse_retriever(chunks)

def get_dense_retriever():
    vector_store = Chroma(persist_directory="./chroma_db", collection_name="greenthumb_kb", embedding_function=OpenAIEmbeddings(model="text-embedding-3-large"))
    return vector_store.as_retriever(search_type="mmr", search_kwargs={'k': 3, 'fetch_k': 10})

def get_sparse_retriever() -> BM25Retriever:
    with open(BM25_PATH, "rb") as f: return pickle.load(f)

def get_hybrid_retriever(query: str, bm25_weight: float = 0.3) -> list[Document]:
    bm25 = get_sparse_retriever()
    dense = get_dense_retriever()
    bm25_results = bm25.invoke(query)
    dense_results = dense.invoke(query)
    scores, seen = {}, {}
    for rank, doc in enumerate(bm25_results):
        key = doc.page_content[:120]
        scores[key] = scores.get(key, 0) + bm25_weight / (rank + 1)
        seen[key] = doc
    for rank, doc in enumerate(dense_results):
        key = doc.page_content[:120]
        scores[key] = scores.get(key, 0) + (1 - bm25_weight) / (rank + 1)
        seen[key] = doc
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [seen[k] for k, _ in ranked[:3]]

llm_rag = ChatOpenAI(model="gpt-4o-mini", temperature=0)
SYSTEM_PROMPT_RAG = (
    "Sei l'assistente clienti di GreenThumb Marketplace, specializzato in giardinaggio e prodotti per l'orto. "
    "Rispondi alle domande dei clienti utilizzando ESCLUSIVAMENTE il contesto fornito dalla knowledge base.\n\n"
    "REGOLE:\n"
    "1. Cita sempre il nome del documento sorgente tra parentesi quadre (es. [spedizioni.md]).\n"
    "2. Se la risposta non è presente nel contesto, rispondi: "
    "'Non ho trovato questa informazione nella nostra knowledge base. "
    "Ti consiglio di contattare il supporto a supporto@greenthumb.it'\n"
    "3. Sii conciso, chiaro e usa un tono cordiale e professionale.\n"
    "4. Per le domande su prodotti, includi sempre il prezzo se disponibile nel contesto.\n\n"
    "CONTESTO:\n{context}"
)
prompt_rag = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT_RAG), ("human", "{input}")])

def _format_docs(docs: list[Document]) -> str:
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "fonte sconosciuta")
        filename = source.split("/")[-1]
        parts.append(f"[{filename}]\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(parts)

def run_rag(question: str) -> dict:
    retrieved_docs = get_hybrid_retriever(question)
    context_str = _format_docs(retrieved_docs)
    chain = prompt_rag | llm_rag | StrOutputParser()
    answer = chain.invoke({"input": question, "context": context_str})
    return {"answer": answer, "context": retrieved_docs}

def get_order_by_id(order_id: str) -> dict | None:
    try:
        with open("./data/orders.json", "r") as f: orders = json.load(f)
        for order in orders:
            if order["order_id"] == order_id: return order
        return None
    except: return None

def create_ticket(customer: dict, order_id: str, issue_type: str, issue_description: str, order_details: dict | None = None) -> dict:
    try:
        with open("./data/tickets.json", "r") as f: tickets = json.load(f)
    except: tickets = []
    try:
        ticket = Ticket(ticket_id=len(tickets) + 1, customer=CustomerInfo(**customer), order_id=order_id, issue_type=IssueType(issue_type), issue_description=issue_description, order_details=OrderDetails(**(order_details or {})))
    except Exception as e: return {"error": str(e)}
    tickets.append(ticket.model_dump(mode="json"))
    with open("./data/tickets.json", "w") as f: json.dump(tickets, f, indent=4, ensure_ascii=False)
    return {"ticket_id": ticket.ticket_id, "status": ticket.status.value, "message": f"Ticket #{ticket.ticket_id} aperto con successo."}

def rag_knowledge_base(query: str) -> dict:
    result = run_rag(query)
    sources = [doc.metadata.get("source", "").split("/")[-1] for doc in result.get("context", [])]
    return {"answer": result.get("answer", ""), "sources": list(dict.fromkeys(sources))}

def execute_tool(name: str, args: dict):
    if name == "get_order_by_id": return get_order_by_id(**args)
    elif name == "create_ticket": return create_ticket(**args)
    elif name == "rag_knowledge_base": return rag_knowledge_base(**args)
    return {"error": "Not found"}

class GreenThumbAgent:
    def __init__(self, model: str = "openai/gpt-4o", max_iterations: int = 5):
        self.model = model
        self.max_iterations = max_iterations
        self.tools = [
            {
                "type": "function",
                "function": {
                    "name": "get_order_by_id",
                    "description": "Recupera i dettagli di un ordine.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "order_id": {"type": "string"}
                        },
                        "required": ["order_id"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "create_ticket",
                    "description": "Apre un ticket di supporto.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "customer": {
                                "type": "object",
                                "properties": {
                                    "name": {"type": "string"},
                                    "email": {"type": "string"},
                                    "phone": {"type": "string"}
                                },
                                "required": ["name", "email", "phone"]
                            },
                            "order_id": {"type": "string"},
                            "issue_type": {"type": "string", "enum": ["MANCATA_CONSEGNA", "PRODOTTO_DANNEGGIATO", "RESO", "RIMBORSO", "ALTRO"]},
                            "issue_description": {"type": "string"},
                            "order_details": {"type": "object"}
                        },
                        "required": ["customer", "order_id", "issue_type", "issue_description"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "rag_knowledge_base",
                    "description": "Cerca nella knowledge base.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "query": {"type": "string"}
                        },
                        "required": ["query"]
                    }
                }
            }
        ]

    def run(self, user_message: str, history: list = None) -> str:
        messages = []
        if history: messages.extend(history)
        else:
            messages.append({
                "role": "system",
                "content": (
                    "Sei l'assistente clienti di GreenThumb. Aiuta il cliente usando i tool disponibili.\n\n"
                    "## USO DELLA KNOWLEDGE BASE (RAG)\n"
                    "→ chiama SEMPRE 'rag_knowledge_base' prima di rispondere su prodotti/policy.\n"
                    "→ Nel campo JSON 'sources' di AgentResponse, copia le fonti del RAG.\n\n"
                    "## REGOLE DI ESCALATION\n"
                    "Apri ticket ed imposta needs_human: true se:\n"
                    "1. Ordine CONSEGNATO non ricevuto.\n"
                    "2. Modifica indirizzo in spedizione (non è possibile, comunica questo poi apri ticket ALTRO).\n"
                    "3. Richiesta reso/rimborso o prodotto danneggiato.\n\n"
                    "## TICKET\n"
                    "STEP 1: Chiama get_order_by_id prima del ticket se hai l'ID ordine.\n"
                    "STEP 2: Chiama create_ticket con i dati dell'ordine.\n"
                )
            })
        messages.append({"role": "user", "content": user_message})
        return self.__react_human_loop__(messages)

    def __react_human_loop__(self, messages: list) -> str:
        for _ in range(self.max_iterations):
            response = completion(model=self.model, messages=messages, tools=self.tools)
            msg = response.choices[0].message
            if msg.tool_calls:
                messages.append(msg)
                for tool_call in msg.tool_calls:
                    res = execute_tool(tool_call.function.name, json.loads(tool_call.function.arguments))
                    messages.append({"role": "tool", "tool_call_id": tool_call.id, "name": tool_call.function.name, "content": json.dumps(res, ensure_ascii=False)})
            else:
                final = completion(model=self.model, messages=messages, response_format=AgentResponse)
                return final.choices[0].message.content
        return json.dumps({"answer": "Timeout", "sources": [], "needs_human": True})
class SummaryBufferHistory(InMemoryChatMessageHistory):
    llm: Any
    max_token_limit: int = 100
    _summary: str = PrivateAttr(default="")
    _enc: Any = PrivateAttr(default=None)

    def __init__(self, llm: Any, max_token_limit: int = 100, **kwargs: Any):
        super().__init__(llm=llm, max_token_limit=max_token_limit, **kwargs)
        object.__setattr__(self, "_enc", tiktoken.encoding_for_model("gpt-3.5-turbo"))
        object.__setattr__(self, "_summary", "")

    def _count_tokens(self) -> int:
        return sum(
            len(self._enc.encode(m.content)) + 4
            for m in self.messages
        )

    def add_message(self, message: BaseMessage) -> None:
        super().add_message(message)
        if self._count_tokens() > self.max_token_limit:
            self._summarize()

    def _summarize(self):
        old_summary = ""
        recent_messages = []
        if not self.messages:
            return
        for m in self.messages:
            if isinstance(m, SystemMessage) and "Riassunto" in m.content:
                old_summary = m.content
            else:
                recent_messages.append(m)

        history_text = "\n".join(
            f"{m.type}:{m.content}" for m in recent_messages
        )
        prompt = f"""Aggiorna il riassunto della conversazione di supporto clienti di GreenThumb Marketplace.
        Riassunto precedente:
        {old_summary}
        Nuovi messaggi di interazione:
        {history_text}

        REGOLE TASSATIVE PER IL NUOVO RIASSUNTO:
        1. Includi l'identità del cliente (Nome, Email o Telefono) se è stata specificata.
        2. Riassumi sinteticamente tutti i dettagli dell'ordine citati (es. ID dell'ordine, se è in spedizione, consegnato o in reso).
        3. Elenca brevemente i prodotti discussi (es. vanga, lavanda, bulbi) e il motivo del contatto (es. richiesta info, ritardo consegna, avvio reso).
        4. Specifica se è stato attivato il tool di escalation o aperto un ticket di supporto.
        5. Formula il riassunto in terza persona in modo fattuale, privo di convenevoli."""
        response = completion(
            model=self.llm,
            messages=[{"role": "user", "content": prompt}]
        )
        object.__setattr__(self, "_summary", response.choices[0].message.content)
        self.clear()
        if self._summary:
            self.messages.append(
                SystemMessage(content=f"Riassunto: {self._summary}")
            )
        logger.info(f"Summary updated: {self._summary}")

_session_store = {}

def get_session_history(session_id: str) -> SummaryBufferHistory:
    if session_id not in _session_store:
        _session_store[session_id] = SummaryBufferHistory(llm="gpt-4o-mini", max_token_limit=150)
    return _session_store[session_id]

def get_formatted_history(session_id: str) -> list[dict]:
    memory = get_session_history(session_id)
    formatted_history = []
    for m in memory.messages:
        if isinstance(m, SystemMessage):
            formatted_history.append({"role": "system", "content": m.content})
        elif isinstance(m, HumanMessage):
            formatted_history.append({"role": "user", "content": m.content})
        elif isinstance(m, AIMessage):
            formatted_history.append({"role": "assistant", "content": m.content})
    return formatted_history

def add_to_history(session_id: str, message: str, role: str = "user"):
    memory = get_session_history(session_id)
    if role == "user":
        memory.add_message(HumanMessage(content=message))
    elif role == "assistant":
        memory.add_message(AIMessage(content=message))


## 4. API (FastAPI) & Web UI (Chainlit) [`src/api`]

In [ ]:
from main import GreenThumbAgent, AgentResponse
from dotenv import load_dotenv
import json
from fastapi import FastAPI
from pydantic import BaseModel

load_dotenv()
agent = GreenThumbAgent()
app = FastAPI()

class ChatRequest(BaseModel):
    message: str

@app.post("/chat/{session_id}")
def chat(session_id: str, chat_request: ChatRequest) -> AgentResponse:
    formatted_history = get_formatted_history(session_id)
    response_json_str = agent.run(user_message=chat_request.message, history=formatted_history)
    try:
        response_dict = json.loads(response_json_str)
        text_answer = response_dict.get("answer", response_json_str)
    except json.JSONDecodeError:
        response_dict = {"answer": response_json_str, "confidence": "low", "sources": [], "needs_human": True}
        text_answer = response_json_str
    add_to_history(session_id, chat_request.message, role="user")
    add_to_history(session_id, text_answer, role="assistant")
    return response_dict

In [ ]:
import chainlit as cl
import uuid
import json
import asyncio
from dotenv import load_dotenv
from main import GreenThumbAgent

load_dotenv()
agent = GreenThumbAgent()

@cl.on_chat_start
async def on_chat_start():
    session_id = str(uuid.uuid4())
    cl.user_session.set("session_id", session_id)
    await cl.Message(content="Benvenuto su GreenThumb! Sono il tuo assistente virtuale.").send()

@cl.on_message
async def on_message(message: cl.Message):
    session_id = cl.user_session.get("session_id")
    user_query = message.content
    msg = cl.Message(content="")
    await msg.send()
    formatted_history = get_formatted_history(session_id)
    response_json_str = await asyncio.to_thread(agent.run, user_message=user_query, history=formatted_history)
    try:
        response_dict = json.loads(response_json_str)
        text_answer = response_dict.get("answer", response_json_str)
        sources = response_dict.get("sources", [])
    except json.JSONDecodeError:
        text_answer = response_json_str
        sources = []
    display_text = text_answer
    if sources:
        display_text += "\n\n*(Fonti: " + ", ".join(sources) + ")*"
    add_to_history(session_id, user_query, role="user")
    add_to_history(session_id, text_answer, role="assistant")
    msg.content = display_text
    await msg.update()

## 5. Script di Valutazione Ragas & DeepEval [`evaluation`]

In [ ]:
#!/usr/bin/env python3
import json
import os
from loguru import logger
from datasets import Dataset

try:
    import sys
    import types
    _vertexai_stub = types.ModuleType("langchain_community.chat_models.vertexai")
    class _ChatVertexAI: pass
    _vertexai_stub.ChatVertexAI = _ChatVertexAI
    sys.modules.setdefault("langchain_community.chat_models.vertexai", _vertexai_stub)

    from ragas import evaluate as ragas_evaluate
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.metrics.collections import answer_relevancy, faithfulness, context_precision, context_recall
    RAGAS_AVAILABLE = True
except Exception as e:
    logger.warning(f"RAGAS non disponibile: {e}")
    RAGAS_AVAILABLE = False

def run_ragas():
    if not RAGAS_AVAILABLE: return
    from main import run_ingestion, run_rag
    run_ingestion()
    with open("evaluation/test_dataset_rag.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for item in eval_data:
        query = item["query"]
        res = run_rag(query)
        ragas_data["question"].append(query)
        ragas_data["answer"].append(res.get("answer", ""))
        ragas_data["contexts"].append([doc.page_content for doc in res.get("context", [])])
        ragas_data["ground_truth"].append(item["ground_truth"])

    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    rl = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
    re = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

    res_eval = ragas_evaluate(
        Dataset.from_dict(ragas_data),
        metrics=[answer_relevancy, faithfulness, context_precision, context_recall],
        llm=rl, embeddings=re
    )
    logger.info(f"Risultati Ragas: {res_eval}")

In [ ]:
run_ragas()

In [ ]:
#!/usr/bin/env python3
import json
from loguru import logger
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate as deepeval_evaluate
from deepeval.evaluate.configs import AsyncConfig, CacheConfig

def run_deepeval():
    run_ingestion()
    with open("evaluation/test_dataset_agent.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    agent = GreenThumbAgent()
    test_cases = []
    for item in eval_data:
        query = item["query"]
        raw_res = agent.run(query)
        try:
            res_obj = json.loads(raw_res)
            act_out = res_obj.get("answer", raw_res)
        except Exception:
            act_out = str(raw_res)
        test_cases.append(LLMTestCase(input=query, actual_output=act_out, expected_output=item["ground_truth"]))

    answer_relevancy = AnswerRelevancyMetric(threshold=0.5, model="gpt-4o")
    correctness = GEval(
        name="Agent Correctness",
        criteria="Valuta se la risposta rispetta le indicazioni (strumenti usati, escalation, tono).",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.5, model="gpt-4o"
    )
    deepeval_evaluate(test_cases, [answer_relevancy, correctness], async_config=AsyncConfig(run_async=True, max_concurrent=1, throttle_value=10))

In [ ]:
run_deepeval()